In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

In [ ]:
n = 100000
p = 0.3
M = 97
trials = 1000
p_values = np.arange(0.1, 0.55, 0.05)

In [ ]:
def estimate_edge_density_er(n, p, M, rng=None):
    """
    Estimate edge density in an Erdős–Rényi G(n, p) graph
    by sampling M random node pairs.

    Parameters
    ----------
    n : int
        Number of nodes
    p : float
        True edge probability
    M : int
        Number of sampled node pairs
    rng : np.random.Generator, optional
        Random number generator for reproducibility

    Returns
    -------
    p_hat : float
        Estimated edge density
    """
    if rng is None:
        rng = np.random.default_rng()

    # Sample Bernoulli(p) outcomes directly
    samples = rng.random(M) < p
    return samples.mean()


def monte_carlo_er(n, p, M, trials=1000, seed=42):
    """
    Run a Monte Carlo simulation to study estimator behavior.

    Returns
    -------
    estimates : np.ndarray
        Array of p_hat estimates from each trial
    """
    rng = np.random.default_rng(seed)
    estimates = np.empty(trials)

    for i in range(trials):
        estimates[i] = estimate_edge_density_er(n, p, M, rng)

    return estimates


In [ ]:
estimates = monte_carlo_er(n, p, M, trials)

print("True p:", p)
print("Mean estimate:", estimates.mean())
print("Bias:", estimates.mean() - p)
print("Std dev:", estimates.std())

In [ ]:
z = norm.ppf(0.975)  # 95% confidence multiplier

means = []
ci_lower = []
ci_upper = []

base_seed = 42       # master seed

for k, p in enumerate(p_values):

    # Independent RNG per p
    rng = np.random.default_rng(base_seed + k)
    
    samples = rng.random((trials, M)) < p
    estimates = samples.mean(axis=1)
    
    mean_est = estimates.mean()
    std_est = estimates.std(ddof=1)

    means.append(mean_est)
    ci = z * std_est
    ci_lower.append(mean_est - ci)
    ci_upper.append(mean_est + ci)

# Plot estimate vs true density
plt.figure()
plt.plot(p_values, means, marker='o', color="blue", label="Mean estimate")
plt.fill_between(p_values, ci_lower, ci_upper, alpha=0.2, label="95% CI")
plt.plot(p_values, p_values, linestyle='--', color="orange", label="True global edge density")
plt.ylim(0, 1)
plt.xlim(0.1, 0.5)
plt.xlabel("True global edge density")
plt.ylabel("Estimated F(G)")
# plt.title("Monte Carlo Estimation of Edge Density (95% CI, M=97)")
plt.legend()
# plt.grid(True)
plt.show()


In [ ]:
z = norm.ppf(0.975)  # 95%
base_seed = 42

coverage_wald = []

for k, p_true in enumerate(p_values):
    rng = np.random.default_rng(base_seed + k)  # independent across p

    # trials x M Bernoulli draws
    samples = (rng.random((trials, M)) < p_true)
    p_hat = samples.mean(axis=1)

    # ---- Wald CI ----
    se = np.sqrt(p_hat * (1 - p_hat) / M)
    wald_lower = p_hat - z * se
    wald_upper = p_hat + z * se
    covered_wald = (wald_lower <= p_true) & (p_true <= wald_upper)
    coverage_wald.append(covered_wald.mean())

coverage_wald = np.array(coverage_wald)

# Plot coverage
plt.figure()
plt.plot(p_values, coverage_wald, marker="o", color="blue", label="Wald 95% CI coverage")
plt.axhline(0.95, linestyle="--", color="orange", label="Nominal 0.95")
plt.ylim(0, 1)
plt.xlim(0.1, 0.5)
plt.xlabel("True global edge density")
plt.ylabel("Coverage probability")
# plt.title(f"Coverage vs p (M={M}, trials={trials})")
plt.legend()
# plt.grid(True)
plt.show()